## Before You Run / 运行前准备

> **Prerequisites — genomes needed for this notebook's tests / 本 notebook 测试所需基因组**
>
> ```bash
> conda activate igem2026
> # phiL7 (required) / 必需
> python 00_raw_data/processes/fetch_phages.py --accession EU717894.1
> # Xcc ATCC 33913 (required) / 必需
> python 00_raw_data/processes/fetch_bacteria.py --accession GCF_000007145.1
> # T7 phage (required for tests only) / 仅测试需要
> python 00_raw_data/processes/fetch_phages.py --accession NC_001604.1
> ```
>
> All three are small downloads (< 10 MB total). Run once; idempotent.
> 三个均为小文件（总计 < 10 MB），运行一次即可，重复运行无副作用。

---

# Module 01 — Fetch Reference Genomes

Downloads reference genomes (Xcc ATCC 33913, phage T7, phage phiL7) from NCBI into
`00_raw_data/{bacteria,phage}/<accession>/` and writes index + manifest files.

Reference: O'Leary, N.A. et al. (2024) *Sci. Data* 11:732 (NCBI Datasets CLI).

---

## 模块 01 — 下载参考基因组

从 NCBI 下载参考基因组（Xcc ATCC 33913、噬菌体 T7、噬菌体 phiL7）到
`00_raw_data/{bacteria,phage}/<accession>/`，并生成索引与 MANIFEST 文件。

参考文献：O'Leary, N.A. et al. (2024) *Sci. Data* 11:732（NCBI Datasets CLI）。

## Cell 1: Library versions
## 第 1 格：库版本信息

In [1]:
import sys, subprocess
import pathlib, csv, hashlib, datetime, shutil, zipfile, time, random

import pandas as pd
import numpy as np
from Bio import SeqIO
import Bio

# Set random seeds for reproducibility / 设置随机种子以确保可重现性
random.seed(42)
np.random.seed(42)

# Print library versions / 打印库版本
print(f"Python      : {sys.version}")
print(f"pandas      : {pd.__version__}")
print(f"numpy       : {np.__version__}")
print(f"biopython   : {Bio.__version__}")
print(f"ncbi-datasets CLI:", end=" ")
result = subprocess.run(["datasets", "--version"], capture_output=True, text=True)
print(result.stdout.strip() or result.stderr.strip())

# Save repo commit sha for metadata / 记录 commit sha 供元数据使用
try:
    REPO_COMMIT_SHA = subprocess.check_output(
        ["git", "rev-parse", "--short", "HEAD"], text=True
    ).strip()
except Exception:
    REPO_COMMIT_SHA = "unknown"
print(f"repo commit : {REPO_COMMIT_SHA}")

Python      : 3.14.0 (v3.14.0:ebf955df7a8, Oct  7 2025, 08:20:14) [Clang 16.0.0 (clang-1600.0.26.6)]
pandas      : 2.3.3
numpy       : 2.3.4
biopython   : 1.87
ncbi-datasets CLI: datasets version: 18.25.1
repo commit : 05a41cd


## Cell 2: Path setup
## 第 2 格：路径设置

Notebooks live in `01_data_ground_truth/processes/`, so `parents[1]` from CWD is repo root.

Notebook 位于 `01_data_ground_truth/processes/`，故从当前工作目录 `parents[1]` 指向 repo 根目录。

In [2]:
# Repo root: notebook lives at <REPO_ROOT>/01_data_ground_truth/processes/
# Repo 根目录：notebook 在 <REPO_ROOT>/01_data_ground_truth/processes/ 下
REPO_ROOT = pathlib.Path.cwd().resolve().parents[1]
print(f"REPO_ROOT   : {REPO_ROOT}")

RAW_DATA    = REPO_ROOT / "00_raw_data"            # raw genome storage / 原始基因组存储
MOD01_DIR   = REPO_ROOT / "01_data_ground_truth"   # this module / 本模块目录
INPUTS_DIR  = MOD01_DIR / "inputs"                 # seed CSV / 种子 CSV
OUTPUTS_DIR = MOD01_DIR / "outputs"                # module outputs / 模块输出

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

# NCBI Entrez email (required by NCBI policy) / NCBI 政策要求提供 email
NCBI_EMAIL = "alexchenworking1.618@gmail.com"

TODAY = datetime.date.today().isoformat()  # YYYY-MM-DD
print(f"Run date    : {TODAY}")

REPO_ROOT   : /Users/alexy/Desktop/Claude Workspace/agent-01-data-ground-truth
Run date    : 2026-05-07


## Cell 3: Helper functions
## 第 3 格：辅助函数

In [3]:
def sha256_file(path: pathlib.Path) -> str:
    """SHA-256 hex digest of a file. / 计算文件 SHA-256 十六进制摘要。"""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()


def count_fasta_records(path: pathlib.Path) -> int:
    """Count FASTA records in a file. / 统计 FASTA 文件中的记录数量。"""
    return sum(1 for _ in SeqIO.parse(str(path), "fasta"))


def append_manifest_row(manifest_path: pathlib.Path, row: dict) -> None:
    """Append a row to MANIFEST.csv (create with header if absent).
    向 MANIFEST.csv 追加一行（若不存在则新建并写表头）。
    """
    fieldnames = [
        "filename", "sha256", "bytes", "n_records",
        "created_utc", "source_acc", "source_module", "notes",
    ]
    write_header = not manifest_path.exists()
    with open(manifest_path, "a", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=fieldnames)
        if write_header:
            writer.writeheader()
        writer.writerow(row)


def run_datasets_download(cmd: list, retries: int = 3) -> bool:
    """Run an NCBI datasets CLI command with exponential backoff on failure.
    使用指数退避策略执行 NCBI datasets CLI 命令。
    """
    delay = 60  # initial wait in seconds / 初始等待秒数
    for attempt in range(1, retries + 1):
        print(f"  Attempt {attempt}/{retries}: {' '.join(cmd)}")
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode == 0:
            return True
        print(f"  WARN: exit {result.returncode} — {result.stderr[:200]}")
        if attempt < retries:
            print(f"  Waiting {delay}s before retry (exponential backoff)...")
            time.sleep(delay)
            delay *= 2  # 60s → 120s → 240s
    return False


def extract_genome_zip(zip_path: pathlib.Path, target_dir: pathlib.Path) -> dict:
    """Extract genome/cds/protein files from a datasets CLI zip archive.
    
    Handles two archive layouts emitted by the datasets CLI:
    - Genome assemblies: files under ncbi_dataset/data/<accession>/
      - *_genomic.fna  → genome.fna
      - cds_from_genomic.fna → cds.fna
      - protein.faa → proteins.faa
    - Virus genomes: files under ncbi_dataset/data/
      - genomic.fna → genome.fna
      - cds.fna → cds.fna
      - protein.faa → proteins.faa
    
    处理 datasets CLI 输出的两种 zip 布局格式：
    - 细菌基因组（assembly）：文件在 ncbi_dataset/data/<accession>/ 下
    - 病毒基因组：文件在 ncbi_dataset/data/ 下
    """
    target_dir.mkdir(parents=True, exist_ok=True)
    extracted = {}

    with zipfile.ZipFile(zip_path) as zf:
        names = zf.namelist()
        print(f"  Zip has {len(names)} entries: {names[:6]}")

        for name in names:
            base = name.split("/")[-1]  # basename of the entry / 条目基本名

            # Genome FASTA — bacteria: *_genomic.fna (not cds_from_genomic.fna)
            #               — virus:   genomic.fna
            # 基因组 FASTA — 细菌：*_genomic.fna（排除 cds_from_genomic.fna）
            #              — 病毒：genomic.fna
            if base == "genomic.fna" or (base.endswith("_genomic.fna") and not base.startswith("cds")):
                dest = target_dir / "genome.fna"
                with zf.open(name) as src, open(dest, "wb") as dst:
                    shutil.copyfileobj(src, dst)
                extracted["genome.fna"] = dest

            # CDS nucleotide sequences
            # CDS 核苷酸序列
            elif base in ("cds.fna", "cds_from_genomic.fna"):
                dest = target_dir / "cds.fna"
                with zf.open(name) as src, open(dest, "wb") as dst:
                    shutil.copyfileobj(src, dst)
                extracted["cds.fna"] = dest

            # Protein sequences
            # 蛋白质序列
            elif base == "protein.faa":
                dest = target_dir / "proteins.faa"
                with zf.open(name) as src, open(dest, "wb") as dst:
                    shutil.copyfileobj(src, dst)
                extracted["proteins.faa"] = dest

    return extracted


print("Helper functions defined. / 辅助函数已定义。")

Helper functions defined. / 辅助函数已定义。


## Cell 4: Load reference_targets.csv
## 第 4 格：读取 reference_targets.csv

In [4]:
targets_path = INPUTS_DIR / "reference_targets.csv"
targets = pd.read_csv(targets_path)
print(f"Loaded {len(targets)} reference targets from {targets_path}")
print(targets.to_string(index=False))

Loaded 3 reference targets from /Users/alexy/Desktop/Claude Workspace/agent-01-data-ground-truth/01_data_ground_truth/inputs/reference_targets.csv
category    assembly_acc nucleotide_acc                                              label          subdir  priority
bacteria GCF_000007145.1     AE008922.1                    Xcc ATCC 33913 (host reference) GCF_000007145.1         1
   phage             NaN    NC_001604.1           Bacteriophage T7 (control phage E. coli)     NC_001604.1         1
   phage             NaN     EU717894.1 phiL7 Xanthomonas campestris phage (main scaffold)      EU717894.1         1


## Cell 5: Fetch or verify each reference genome
## 第 5 格：下载或验证每个参考基因组

Download strategy:
- **bacteria** (GCF accession): `datasets download genome accession <GCF> --include genome,cds,protein`
- **phage/virus** (RefSeq nucleotide): `datasets download virus genome accession <NC_> --include genome,cds,protein`
- If all three target files (`genome.fna`, `cds.fna`, `proteins.faa`) already exist: **verify only**, no re-download.
- If any file is missing: re-download.

下载策略：
- **细菌**（GCF 登录号）：`datasets download genome accession`
- **噬菌体/病毒**（RefSeq 核苷酸）：`datasets download virus genome accession`
- 若三个目标文件均存在：**仅验证**，不重复下载。
- 若有文件缺失：重新下载。

**Note:** Bio.Entrez SSL fails on this machine (corporate proxy self-signed cert).
All downloads use `datasets` CLI which has its own TLS implementation.

**注意：** 本机 Bio.Entrez SSL 连接失败（企业代理自签名证书）。
所有下载改走 `datasets` CLI，其 TLS 实现不受影响。

In [5]:
fetch_log = []   # rows for download_log CSV / 下载日志行
index_rows = []  # rows for reference_genomes_index CSV / 索引文件行

for _, tgt in targets.iterrows():
    category     = tgt["category"]        # 'phage' or 'bacteria'
    assembly_acc = str(tgt["assembly_acc"]) if pd.notna(tgt["assembly_acc"]) else ""
    nucl_acc     = str(tgt["nucleotide_acc"]) if pd.notna(tgt["nucleotide_acc"]) else ""
    label        = tgt["label"]
    subdir       = tgt["subdir"]

    primary_acc = assembly_acc if assembly_acc else nucl_acc
    target_dir  = RAW_DATA / category / subdir

    # Check which of the three expected files are already present
    # 检查三个预期文件是否已经存在
    expected_files = ["genome.fna", "cds.fna", "proteins.faa"]
    present = {f: (target_dir / f).exists() for f in expected_files}
    all_present = all(present.values())

    print(f"\n{'='*60}")
    print(f"Target : {label}")
    print(f"Acc    : {primary_acc}  →  {target_dir.relative_to(REPO_ROOT)}")
    print(f"Files  : { {k: '✓' if v else '✗' for k, v in present.items()} }")

    now_utc = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

    # ── Verify-only if all files exist ───────────────────────────────────────
    # 若所有文件均存在则跳过下载
    if all_present:
        genome_len = sum(len(r.seq) for r in SeqIO.parse(
            str(target_dir / "genome.fna"), "fasta"
        ))
        sha = sha256_file(target_dir / "genome.fna")
        print(f"  ✓ All files present — genome length = {genome_len:,} bp")
        fetch_log.append({
            "accession": primary_acc, "status": "already_present",
            "attempts": 0, "error_msg": "", "fetched_utc": now_utc,
        })
        index_rows.append({
            "label": label, "accession": primary_acc,
            "target_dir": str(target_dir.relative_to(REPO_ROOT)),
            "status": "verified", "sha256_genome": sha,
        })
        continue

    # ── Download ─────────────────────────────────────────────────────────────
    # 执行下载
    target_dir.mkdir(parents=True, exist_ok=True)
    zip_path = target_dir / "ncbi_dataset.zip"

    if category == "bacteria":
        cmd = [
            "datasets", "download", "genome", "accession", assembly_acc,
            "--include", "genome,cds,protein",
            "--filename", str(zip_path),
            "--no-progressbar",
        ]
    else:  # phage / virus
        cmd = [
            "datasets", "download", "virus", "genome", "accession", nucl_acc,
            "--include", "genome,cds,protein",
            "--filename", str(zip_path),
        ]

    ok = run_datasets_download(cmd)
    attempts = 1 if ok else 3
    error_msg = "" if ok else "Download failed after 3 retries"

    if ok and zip_path.exists():
        print(f"  Downloaded {zip_path.stat().st_size:,} bytes")
        extracted = extract_genome_zip(zip_path, target_dir)
        zip_path.unlink()  # save disk space / 删除 zip 节省空间
        print(f"  Extracted: {list(extracted.keys())}")

        genome_fna = target_dir / "genome.fna"
        if genome_fna.exists():
            genome_len = sum(len(r.seq) for r in SeqIO.parse(str(genome_fna), "fasta"))
            sha = sha256_file(genome_fna)
            print(f"  ✓ genome.fna — {genome_len:,} bp")
            status = "downloaded"
        else:
            sha, status = "", "error"
            error_msg = "genome.fna missing after extraction"
            print(f"  ✗ genome.fna not found after extraction!")
    else:
        sha, status = "", "error"
        print(f"  ✗ Download failed")

    fetch_log.append({
        "accession": primary_acc, "status": status,
        "attempts": attempts, "error_msg": error_msg, "fetched_utc": now_utc,
    })
    index_rows.append({
        "label": label, "accession": primary_acc,
        "target_dir": str(target_dir.relative_to(REPO_ROOT)),
        "status": status, "sha256_genome": sha,
    })

print("\nFetch loop complete. / 下载循环完成。")


Target : Xcc ATCC 33913 (host reference)
Acc    : GCF_000007145.1  →  00_raw_data/bacteria/GCF_000007145.1
Files  : {'genome.fna': '✓', 'cds.fna': '✓', 'proteins.faa': '✓'}
  ✓ All files present — genome length = 5,076,188 bp

Target : Bacteriophage T7 (control phage E. coli)
Acc    : NC_001604.1  →  00_raw_data/phage/NC_001604.1
Files  : {'genome.fna': '✓', 'cds.fna': '✓', 'proteins.faa': '✓'}
  ✓ All files present — genome length = 39,937 bp

Target : phiL7 Xanthomonas campestris phage (main scaffold)
Acc    : EU717894.1  →  00_raw_data/phage/EU717894.1
Files  : {'genome.fna': '✓', 'cds.fna': '✓', 'proteins.faa': '✓'}
  ✓ All files present — genome length = 44,080 bp

Fetch loop complete. / 下载循环完成。


## Cell 6: Write download_log and reference_genomes_index
## 第 6 格：写入下载日志和参考基因组索引

In [6]:
# Download log / 下载日志
log_path = OUTPUTS_DIR / f"download_log_{TODAY}.csv"
pd.DataFrame(fetch_log).to_csv(log_path, index=False)
print(f"Wrote download log: {log_path}")
print(pd.DataFrame(fetch_log).to_string(index=False))

# Reference genomes index / 参考基因组索引
index_path = OUTPUTS_DIR / "reference_genomes_index.csv"
pd.DataFrame(index_rows).to_csv(index_path, index=False)
print(f"\nWrote index: {index_path}")
print(pd.DataFrame(index_rows).to_string(index=False))

Wrote download log: /Users/alexy/Desktop/Claude Workspace/agent-01-data-ground-truth/01_data_ground_truth/outputs/download_log_2026-05-07.csv
      accession          status  attempts error_msg          fetched_utc
GCF_000007145.1 already_present         0           2026-05-08T05:28:47Z
    NC_001604.1 already_present         0           2026-05-08T05:28:47Z
     EU717894.1 already_present         0           2026-05-08T05:28:47Z

Wrote index: /Users/alexy/Desktop/Claude Workspace/agent-01-data-ground-truth/01_data_ground_truth/outputs/reference_genomes_index.csv
                                             label       accession                           target_dir   status                                                    sha256_genome
                   Xcc ATCC 33913 (host reference) GCF_000007145.1 00_raw_data/bacteria/GCF_000007145.1 verified 68456fd80a2d1a7dd781403f5c32e0af4dce8782a26fb9a45998c4cdfff2fbd7
          Bacteriophage T7 (control phage E. coli)     NC_001604.1        

## Cell 7: Write Module 01 MANIFEST.csv and update global 00_raw_data MANIFEST
## 第 7 格：写入模块 MANIFEST.csv 并更新全局 00_raw_data MANIFEST

INTERFACE.md requires every `outputs/` directory to have a MANIFEST.csv with columns:
`filename, sha256, bytes, n_records, created_utc, source_acc, source_module, notes`.

We also append new entries to `00_raw_data/MANIFEST.csv` (the global checksum index for
all raw genome files).

INTERFACE.md 要求每个 `outputs/` 目录有 MANIFEST.csv，列名为：
`filename, sha256, bytes, n_records, created_utc, source_acc, source_module, notes`。

同时向全局 `00_raw_data/MANIFEST.csv` 追加新条目（避免重复）。

In [7]:
mod01_manifest = OUTPUTS_DIR / "MANIFEST.csv"
global_manifest = RAW_DATA / "MANIFEST.csv"
NOW_UTC = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

# Fresh module manifest each run / 每次运行重新生成模块 manifest
if mod01_manifest.exists():
    mod01_manifest.unlink()

# Load existing global manifest to avoid duplicate entries
# 读取全局 manifest 以避免重复条目
existing_global_keys: set = set()
if global_manifest.exists():
    try:
        existing_df = pd.read_csv(global_manifest)
        existing_global_keys = set(existing_df["filename"].tolist())
    except Exception:
        pass

for row in index_rows:
    acc = row["accession"]
    target_dir_abs = REPO_ROOT / row["target_dir"]
    cat = "bacteria" if "bacteria" in row["target_dir"] else "phage"

    for fname in ["genome.fna", "cds.fna", "proteins.faa"]:
        fpath = target_dir_abs / fname
        if not fpath.exists():
            continue
        sha   = sha256_file(fpath)
        fsize = fpath.stat().st_size
        try:
            n_rec = count_fasta_records(fpath)
        except Exception:
            n_rec = ""

        manifest_row = {
            "filename":      f"{cat}/{acc}/{fname}",
            "sha256":        sha,
            "bytes":         fsize,
            "n_records":     n_rec,
            "created_utc":   NOW_UTC,
            "source_acc":    acc,
            "source_module": "01_data_ground_truth",
            "notes":         row["label"],
        }
        # Module MANIFEST / 模块 MANIFEST
        append_manifest_row(mod01_manifest, manifest_row)

        # Global MANIFEST — skip if already present / 全局 MANIFEST — 已存在则跳过
        key = manifest_row["filename"]
        if key not in existing_global_keys:
            append_manifest_row(global_manifest, manifest_row)
            existing_global_keys.add(key)

n_rows = sum(1 for _ in open(mod01_manifest)) - 1  # subtract header line / 减去表头行
print(f"Module MANIFEST: {mod01_manifest} ({n_rows} file rows)")
print(pd.read_csv(mod01_manifest).to_string(index=False))

Module MANIFEST: /Users/alexy/Desktop/Claude Workspace/agent-01-data-ground-truth/01_data_ground_truth/outputs/MANIFEST.csv (9 file rows)
                             filename                                                           sha256   bytes  n_records          created_utc      source_acc        source_module                                              notes
  bacteria/GCF_000007145.1/genome.fna 68456fd80a2d1a7dd781403f5c32e0af4dce8782a26fb9a45998c4cdfff2fbd7 5139727          1 2026-05-08T05:28:48Z GCF_000007145.1 01_data_ground_truth                    Xcc ATCC 33913 (host reference)
     bacteria/GCF_000007145.1/cds.fna 935dd8459948f9cbb02cab07af209aa9acb388e2bc173ff91569d0e644e318c1 5196419       4276 2026-05-08T05:28:48Z GCF_000007145.1 01_data_ground_truth                    Xcc ATCC 33913 (host reference)
bacteria/GCF_000007145.1/proteins.faa cb7fe247c73e3bf6122303eddf02b3e5b4c56a860a39b33bd319f90e83cdc9c3 1725195       4093 2026-05-08T05:28:48Z GCF_000007145.1 01_data_gr

## Cell 8: Verification cell
## 第 8 格：验证格

Assert genome lengths are within literature-expected ranges. Notebook fails here if any
downloaded genome is implausible.

断言基因组长度在文献预期范围内。若有下载的基因组不合理，本格将报错终止。

In [8]:
# Expected genome length ranges (bp) from literature
# 文献记载的预期基因组长度范围（bp）
EXPECTED_LENGTHS = {
    "EU717894.1":     (44_000, 46_000, "phiL7 — Lee et al. 2009 AEM 75:7828"),
    "NC_001604.1":    (39_800, 40_100, "T7 phage — Studier lab (RefSeq NC_001604.1)"),
    "GCF_000007145.1": (5_050_000, 5_100_000, "Xcc ATCC 33913 — da Silva et al. 2002 Nature 417:459"),
}

all_ok = True
for row in index_rows:
    acc        = row["accession"]
    genome_fna = REPO_ROOT / row["target_dir"] / "genome.fna"
    if not genome_fna.exists():
        print(f"  SKIP {acc}: genome.fna not found")
        continue

    total_len = sum(len(r.seq) for r in SeqIO.parse(str(genome_fna), "fasta"))
    lo, hi, note = EXPECTED_LENGTHS.get(acc, (1, 999_999_999, "no expectation set"))
    ok = lo <= total_len <= hi
    print(f"  {'✓' if ok else '✗'} {acc}: {total_len:,} bp  [{note}]")
    if not ok:
        all_ok = False
        print(f"      FAIL: expected {lo:,}–{hi:,} bp")

assert all_ok, "One or more genomes have implausible lengths — check download!"
print("\n✓ All genome lengths verified. / 所有基因组长度验证通过。")

  ✓ GCF_000007145.1: 5,076,188 bp  [Xcc ATCC 33913 — da Silva et al. 2002 Nature 417:459]
  ✓ NC_001604.1: 39,937 bp  [T7 phage — Studier lab (RefSeq NC_001604.1)]
  ✓ EU717894.1: 44,080 bp  [phiL7 — Lee et al. 2009 AEM 75:7828]

✓ All genome lengths verified. / 所有基因组长度验证通过。
